# Tutorial 3 (T3): Cross-Validation in Python
**Course / Lab Project**: Precision AgTech Model Evaluation & Validation
**Goal**: Evaluate model performance and stability using K-Fold and Stratified K-Fold Cross Validation.

---
## Objectives of Tutorial 3
1. **Understanding K-Fold Cross Validation**: Dividing dataset into $K$ equal folds to prevent train-test split bias.
2. **Stratified K-Fold CV**: Ensuring class balance across folds for multi-class crop prediction.
3. **Multi-Algorithm Benchmark**: Comparing Random Forest, Decision Tree, Support Vector Machine (SVM), K-Nearest Neighbors (KNN), Gaussian Naive Bayes, and Logistic Regression.


In [ ]:
# Step 1: Import Cross Validation modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import os

print("Cross-Validation libraries imported!")

## Step 1: Load and Preprocess Data

In [ ]:
file_path = os.path.join("data", "Crop_recommendation.csv")
if not os.path.exists(file_path):
    file_path = os.path.join("..", "data", "Crop_recommendation.csv")

df = pd.read_csv(file_path)
feature_cols = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
X = df[feature_cols]
y_raw = df["label"]

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Data ready: X={X_scaled.shape}, y={y.shape}")

## Step 2: 5-Fold Stratified Cross-Validation Benchmark

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
classifiers = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM (RBF Kernel)": SVC(kernel="rbf", random_state=42),
    "KNN (K=5)": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

results = []
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_scaled, y, cv=skf, scoring="accuracy") * 100
    results.append({
        "Algorithm": name,
        "Mean Accuracy (%)": scores.mean(),
        "Std Dev (%)": scores.std(),
        "Min Fold (%)": scores.min(),
        "Max Fold (%)": scores.max()
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="Mean Accuracy (%)", ascending=False)